# Announced vs executed: Form 144 meets Form 4

A Form 144 announces an intent to sell restricted stock; the Form 4
reports what actually traded. Crossing the two families shows how much
of what insiders announce gets executed, and how fast.

Needs `pandas` and `matplotlib`, and a free API key in
`THREESPREAD_API_KEY` ([signup](https://3spread.com/auth/signup)).

In [1]:
import datetime as dt

import pandas as pd

from py3spread import Client

TICKER = "AAPL"
client = Client()
end = dt.date.today()
start = end - dt.timedelta(days=545)

notices = pd.DataFrame(client.proposed_sales.iter(
    ticker=TICKER, accepted_start=str(start), accepted_end=str(end)))
# nothing_to_report_past_3_months refers to table II (prior sales), so it
# does not disqualify the proposed sale itself
notices["units"] = pd.to_numeric(notices["no_of_units_sold"], errors="coerce")
notices["notice_date"] = pd.to_datetime(notices["accepted_time"].str[:10])
print(f"{len(notices)} sale notices")

executions = {}
for t in client.insiders.iter_transactions(
    issuer_ticker=TICKER, transaction_kind="nonderiv",
    transaction_acquired_disposed_code="D",
    transaction_start=str(start), transaction_end=str(end),
):
    accession = t["filing_id"].split("_", 1)[1]
    executions[(accession, t["record_index"])] = t
sales = pd.DataFrame(executions.values())
sales["transaction_date"] = pd.to_datetime(sales["transaction_date"])
sales["shares"] = pd.to_numeric(sales["transaction_shares"], errors="coerce")
print(f"{len(sales)} executed sale records")

17 sale notices
68 executed sale records


Join on the person: Form 144 carries `seller_name`, Form 4 carries
`filer_name`, same normalized casing on both sides. For each notice,
find executions by that seller in the following 30 days.

In [2]:
def norm(s):
    return " ".join(str(s).upper().split())

sales["who"] = sales["filer_name"].map(norm)

rows = []
for n in notices.itertuples():
    who = norm(n.seller_name)
    window = sales[(sales["who"] == who)
                   & (sales["transaction_date"] >= n.notice_date)
                   & (sales["transaction_date"] <= n.notice_date + pd.Timedelta(days=30))]
    executed = window["shares"].sum()
    rows.append({
        "notice_date": str(n.notice_date.date()),
        "seller": n.seller_name,
        "announced": n.units,
        "executed_30d": executed,
        "follow_through": executed / n.units if n.units else None,
        "days_to_first": (window["transaction_date"].min() - n.notice_date).days
                         if len(window) else None,
    })

result = pd.DataFrame(rows).sort_values("notice_date", ascending=False)
matched = result["follow_through"].notna() & (result["follow_through"] > 0)
print(f"{matched.sum()} of {len(result)} notices show executions within 30 days")
(result.head(12).style.format({"announced": "{:,.0f}", "executed_30d": "{:,.0f}",
                               "follow_through": "{:.0%}", "days_to_first": "{:.0f}"},
                              na_rep="").hide(axis="index"))

10 of 17 notices show executions within 30 days


notice_date,seller,announced,executed_30d,follow_through,days_to_first
2026-05-06,LEVINSON ARTHUR D,"250,000","255,000",102%,0
2026-05-05,KATHERINE L ADAMS,"43,000",0,0%,
2026-04-23,KEVAN PAREKH,"1,534",0,0%,
2026-04-02,O'BRIEN DEIRDRE,"30,002","30,002",100%,0
2026-04-02,COOK TIMOTHY D,"64,949","64,949",100%,0
2025-10-16,KEVAN PAREKH,"4,199",0,0%,
2025-10-02,KATHERINE L ADAMS,"47,125",0,0%,
2025-10-02,COOK TIMOTHY D,"129,962","129,963",100%,0
2025-10-02,WILLIAMS JEFFREY E,"43,013",0,0%,
2025-10-02,O'BRIEN DEIRDRE,"43,013","43,013",100%,0


Notices without a match are not necessarily unexecuted: a 144 is good
for 90 days and the seller may trade later, in smaller pieces, or under
a different reporting name. Widen the window or loosen the name match
to trade precision for recall.